# 🚀 Dịch Máy Toàn Diện (Full Dataset) với Seq2Seq Attention & Tối Ưu CUDA Toolkit
Notebook này được thiết kế chuyên biệt để:
1. **Rà soát & Huấn luyện trên toàn bộ file dữ liệu gốc:** `datasetTMD.tsv` (~9GB).
2. **Tối ưu hóa tối đa phần cứng bằng CUDA Toolkit:** Kích hoạt nhân **Tensor Cores (AMP FP16)**, **CUDA Pinned Memory**, và **cuDNN Benchmark** trên GPU RTX 4070.
3. **Tích hợp Bộ Tách Từ Tiếng Việt (Word Segmentation / PyVi):** Giúp mô hình hiểu trọn vẹn từ ghép tiếng Việt thay vì chém rời rạc từng âm tiết.
4. **Cấu trúc phân tách chi tiết:** Mỗi hàm, mỗi thuật toán, mỗi epoch nằm ở **từng cell riêng biệt** để bạn dễ dàng chạy, kiểm tra và debug từng phần.

## 1. Cấu hình Phần cứng & Kích hoạt Tối ưu hóa CUDA Toolkit
Thiết lập thiết bị `cuda`, bật `cudnn.benchmark = True` để cuDNN tự động chọn kernel tính toán nhanh nhất trên GPU NVIDIA, và cố định Seed ngẫu nhiên.

In [1]:
import os
import sys
import time
import math
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# 1. Cố định Seed ngẫu nhiên để tái lập kết quả
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 2. Thiết lập thiết bị và KÍCH HOẠT TỐI ƯU CUDA TOOLKIT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    # Kích hoạt cuDNN Benchmark: Tự động đo và chọn kernel CUDA nhanh nhất cho GPU
    torch.backends.cudnn.benchmark = True
    
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cuda_ver = torch.version.cuda
    print("=" * 65)
    print("⚡ ĐÃ KÍCH HOẠT TỐI ƯU HÓA CUDA TOOLKIT THÀNH CÔNG:")
    print(f"🔹 Thiết bị         : {device.type.upper()} ({gpu_name})")
    print(f"🔹 Dung lượng VRAM  : {vram_gb:.2f} GB")
    print(f"🔹 Phiên bản CUDA   : {cuda_ver}")
    print(f"🔹 cuDNN Benchmark  : {torch.backends.cudnn.benchmark} (Tối đa hóa tốc độ Kernel)")
    print("=" * 65)
else:
    print("⚠️ Cảnh báo: Không tìm thấy CUDA GPU, đang chạy trên CPU!")

⚡ ĐÃ KÍCH HOẠT TỐI ƯU HÓA CUDA TOOLKIT THÀNH CÔNG:
🔹 Thiết bị         : CUDA (NVIDIA GeForce RTX 4070 Laptop GPU)
🔹 Dung lượng VRAM  : 8.00 GB
🔹 Phiên bản CUDA   : 12.6
🔹 cuDNN Benchmark  : True (Tối đa hóa tốc độ Kernel)


## 2. Rà soát & Khám phá File Dữ Liệu 9GB (datasetTMD.tsv)
Rà soát nhanh file dữ liệu gốc để kiểm tra dung lượng, số lượng dòng, định dạng các cột và xem thử các cặp câu mẫu đầu tiên.

In [2]:
raw_data_path = os.path.join("MachineTranslation_en_vi", "datasetTMD.tsv")

print("=" * 65)
print("🔍 BƯỚC RÀ SOÁT TỔNG THỂ DỮ LIỆU:")
print("=" * 65)

if os.path.exists(raw_data_path):
    file_size_bytes = os.path.getsize(raw_data_path)
    file_size_gb = file_size_bytes / (1024 ** 3)
    print(f"📁 Đường dẫn file  : {raw_data_path}")
    print(f"💾 Dung lượng file : {file_size_gb:.2f} GB ({file_size_bytes:,} bytes)")
    
    # Đọc nhanh 5 dòng đầu tiên để xem cấu trúc thực tế
    print("\n📄 5 Cặp câu mẫu đầu tiên trong file:")
    with open(raw_data_path, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip()
        print(f"   [Cột tiêu đề]: {header}")
        for i in range(5):
            line = f.readline().strip()
            parts = line.split("\t")
            if len(parts) == 2:
                print(f"   Mẫu {i+1} | EN: {parts[0][:40]:<40} | VI: {parts[1][:40]}")
    print("=" * 65)
else:
    print(f"❌ Không tìm thấy file: {raw_data_path}")

🔍 BƯỚC RÀ SOÁT TỔNG THỂ DỮ LIỆU:
📁 Đường dẫn file  : MachineTranslation_en_vi\datasetTMD.tsv
💾 Dung lượng file : 8.47 GB (9,089,361,507 bytes)

📄 5 Cặp câu mẫu đầu tiên trong file:
   [Cột tiêu đề]: en	vi
   Mẫu 1 | EN: Support quickly.                         | VI: Hỗ trợ nhanh chóng.
   Mẫu 2 | EN: Time Consumption                         | VI: Sự tiêu thụ thời gian
   Mẫu 3 | EN: Fuel Consumption                         | VI: Sự tiêu thụ xăng dầu
   Mẫu 4 | EN: Current Consumption                      | VI: Mức tiêu thụ hiện tại
   Mẫu 5 | EN: Discuss books.                           | VI: Thảo luận về sách.


## 3. Tích hợp Bộ Tách Từ Tiếng Việt (PyVi Tokenization)
Sử dụng thư viện `pyvi` để nhận diện và nối các **từ ghép tiếng Việt** bằng dấu gạch dưới (ví dụ: `đọc sách` $\rightarrow$ `đọc_sách`, `thư giãn` $\rightarrow$ `thư_giãn`), giúp mô hình bảo toàn trọn vẹn ngữ nghĩa.

In [3]:
from pyvi import ViTokenizer

def tokenize_vietnamese(text):
    """Tách từ ghép tiếng Việt bằng PyVi rồi tách token"""
    text = str(text).lower().strip()
    # 1. Nối từ ghép bằng PyVi (vd: 'học sinh' -> 'học_sinh')
    text_segmented = ViTokenizer.tokenize(text)
    # 2. Tách từ bằng regex giữ lại chữ và dấu câu
    tokens = re.findall(r"\w+|[^\w\s]", text_segmented, re.UNICODE)
    return tokens

def tokenize_english(text):
    """Tách từ tiếng Anh bằng Regex chuẩn"""
    text = str(text).lower().strip()
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    return tokens

# Thử nghiệm bộ tách từ trên câu thực tế
sample_vi = "Tôi có niềm đam mê đọc sách và thư giãn vào buổi tối."
sample_en = "I have a big passion for reading books."

print("🧪 KIỂM TRA BỘ TÁCH TỪ TIẾNG VIỆT:")
print(f"Câu gốc        : {sample_vi}")
print(f"Sau khi tách từ : {tokenize_vietnamese(sample_vi)}")
print("\n🧪 KIỂM TRA BỘ TÁCH TỪ TIẾNG ANH:")
print(f"Câu gốc        : {sample_en}")
print(f"Sau khi tách từ : {tokenize_english(sample_en)}")

🧪 KIỂM TRA BỘ TÁCH TỪ TIẾNG VIỆT:
Câu gốc        : Tôi có niềm đam mê đọc sách và thư giãn vào buổi tối.
Sau khi tách từ : ['tôi', 'có', 'niềm', 'đam_mê', 'đọc', 'sách', 'và', 'thư', 'giãn', 'vào', 'buổi', 'tối', '.']

🧪 KIỂM TRA BỘ TÁCH TỪ TIẾNG ANH:
Câu gốc        : I have a big passion for reading books.
Sau khi tách từ : ['i', 'have', 'a', 'big', 'passion', 'for', 'reading', 'books', '.']


## 4. Định nghĩa Lớp Bộ Từ Vựng (Vocabulary)
Lớp `Vocabulary` hỗ trợ ánh xạ 2 chiều giữa Từ và Số, quản lý 4 token đặc biệt (`<pad>`, `<sos>`, `<eos>`, `<unk>`) và lọc bỏ từ hiếm xuất hiện dưới `min_freq` lần.

In [4]:
class Vocabulary:
    def __init__(self, name="vocab", max_vocab_size=32000, min_freq=3, tokenizer_func=None):
        self.name = name
        self.max_vocab_size = max_vocab_size
        self.min_freq = min_freq
        self.tokenizer = tokenizer_func
        
        # Bảng ánh xạ 2 chiều
        self.word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.word_counts = Counter()
        
        self.PAD_IDX = 0
        self.SOS_IDX = 1
        self.EOS_IDX = 2
        self.UNK_IDX = 3

    def build_vocab(self, sentence_list):
        print(f"⏳ Đang đếm tần suất từ cho {self.name}...")
        for sent in sentence_list:
            tokens = self.tokenizer(sent)
            self.word_counts.update(tokens)

        # Lấy top các từ phổ biến nhất theo max_vocab_size và min_freq
        idx = 4
        frequent_words = [
            (w, c) for w, c in self.word_counts.most_common() if c >= self.min_freq
        ]
        # Giới hạn kích thước tối đa để không bao giờ bị tràn VRAM / RAM
        selected_words = frequent_words[:self.max_vocab_size]

        for word, count in selected_words:
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            idx += 1
                
        print(f"✅ Xây dựng xong {self.name}: {len(self.word2idx):,} từ (Đã khống chế tối đa {self.max_vocab_size:,} từ để tối ưu bộ nhớ)")

    def numericalize(self, text):
        tokens = self.tokenizer(text)
        return [self.word2idx.get(t, self.UNK_IDX) for t in tokens]

    def decode(self, indices):
        words = []
        for idx in indices:
            if idx in (self.PAD_IDX, self.SOS_IDX):
                continue
            if idx == self.EOS_IDX:
                break
            words.append(self.idx2word.get(idx, "<unk>"))
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

## 5. Nạp Toàn Bộ Dữ Liệu & Xây Dựng Bộ Từ Điển Toàn Cầu
Đọc toàn bộ file dữ liệu `datasetTMD.tsv`, lọc sạch các câu lỗi/rỗng/quá lệch độ dài và xây dựng từ điển `vocab_en` và `vocab_vi`.

In [5]:
# Đọc và lọc sạch toàn bộ dữ liệu từ file 9GB
clean_pairs = []
seen_en = set()

print(f"⏳ Đang nạp và lọc dữ liệu từ {raw_data_path}...")
start_time = time.time()

with open(raw_data_path, "r", encoding="utf-8", errors="ignore") as f:
    header = f.readline()
    for idx, line in enumerate(f):
        parts = line.strip().split("\t")
        if len(parts) != 2:
            continue
        en, vi = parts[0].strip(), parts[1].strip()
        if not en or not vi:
            continue
        if en in seen_en:
            continue

        en_words = en.split()
        vi_words = vi.split()
        len_en, len_vi = len(en_words), len(vi_words)

        # Lọc độ dài câu hợp lý cho dịch máy
        if 2 <= len_en <= 50 and 2 <= len_vi <= 60:
            ratio = len_vi / len_en
            if 0.35 <= ratio <= 2.8:
                clean_pairs.append((en, vi))
                seen_en.add(en)

elapsed = time.time() - start_time
print(f"✅ Đã nạp thành công {len(clean_pairs):,} cặp câu sạch trong {elapsed:.2f}s!")

# Chuyển đổi sang DataFrame
df_all = pd.DataFrame(clean_pairs, columns=["en", "vi"])

# Khởi tạo và xây dựng 2 bộ từ điển (Khống chế 32.000 từ chuẩn NMT để tối ưu bộ nhớ)
vocab_en = Vocabulary(name="Từ điển Tiếng Anh", max_vocab_size=32000, min_freq=3, tokenizer_func=tokenize_english)
vocab_vi = Vocabulary(name="Từ điển Tiếng Việt (PyVi)", max_vocab_size=32000, min_freq=3, tokenizer_func=tokenize_vietnamese)

vocab_en.build_vocab(df_all['en'])
vocab_vi.build_vocab(df_all['vi'])

⏳ Đang nạp và lọc dữ liệu từ MachineTranslation_en_vi\datasetTMD.tsv...
✅ Đã nạp thành công 26,075,562 cặp câu sạch trong 136.53s!
⏳ Đang đếm tần suất từ cho Từ điển Tiếng Anh...
✅ Xây dựng xong Từ điển Tiếng Anh: 32,004 từ (Đã khống chế tối đa 32,000 từ để tối ưu bộ nhớ)
⏳ Đang đếm tần suất từ cho Từ điển Tiếng Việt (PyVi)...
✅ Xây dựng xong Từ điển Tiếng Việt (PyVi): 32,004 từ (Đã khống chế tối đa 32,000 từ để tối ưu bộ nhớ)


## 6. Định nghĩa Lớp PyTorch Dataset
Lớp `TranslationDataset` chịu trách nhiệm chuyển văn bản thành vector chỉ số số nguyên, tự động chèn `<sos>` vào đầu và `<eos>` vào cuối câu.

In [6]:
class TranslationDataset(Dataset):
    def __init__(self, df, vocab_src, vocab_tgt):
        self.src_sentences = df['en'].tolist()
        self.tgt_sentences = df['vi'].tolist()
        self.vocab_src = vocab_src
        self.vocab_tgt = vocab_tgt

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_text = self.src_sentences[idx]
        tgt_text = self.tgt_sentences[idx]

        # Nguồn: <sos> + tokens + <eos>
        src_indices = [self.vocab_src.SOS_IDX] + self.vocab_src.numericalize(src_text) + [self.vocab_src.EOS_IDX]
        
        # Đích: <sos> + tokens + <eos>
        tgt_indices = [self.vocab_tgt.SOS_IDX] + self.vocab_tgt.numericalize(tgt_text) + [self.vocab_tgt.EOS_IDX]

        return torch.tensor(src_indices, dtype=torch.long), torch.tensor(tgt_indices, dtype=torch.long)

## 7. Định nghĩa Hàm Ghép Batch Động (Dynamic Collate Function)
Hàm `collate_fn` thực hiện đệm `<pad>` tự động sao cho tất cả câu trong cùng một mini-batch có cùng độ dài lớn nhất của batch đó.

In [7]:
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    
    # Pad câu nguồn bằng PAD_IDX (0)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0)
    
    # Pad câu đích bằng PAD_IDX (0)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    
    return src_padded, tgt_padded

## 8. Phân Chia Dữ Liệu & Tạo DataLoader với CUDA Pinned Memory
Phân chia Train (90%), Validation (5%), Test (5%). Kích hoạt `pin_memory = True` để tăng tốc truyền dữ liệu từ RAM CPU lên VRAM GPU qua PCIe.

In [8]:
# 1. Phân chia tập dữ liệu
train_df, temp_df = train_test_split(df_all[['en', 'vi']], test_size=0.10, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, shuffle=True)

# 2. Khởi tạo Datasets
train_dataset = TranslationDataset(train_df, vocab_en, vocab_vi)
val_dataset = TranslationDataset(val_df, vocab_en, vocab_vi)
test_dataset = TranslationDataset(test_df, vocab_en, vocab_vi)

# 3. Kích thước Batch chuẩn cho GPU RTX 4070 (64 câu/batch, tiêu tốn chưa đến 500MB VRAM)
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    pin_memory=True   # Tối ưu CUDA: Truyền DMA siêu tốc lên VRAM
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn, 
    pin_memory=True
)

print(f"📦 Train Loader : {len(train_loader):,} batches ({len(train_dataset):,} câu)")
print(f"📦 Val Loader   : {len(val_loader):,} batches ({len(val_dataset):,} câu)")
print(f"📦 Test Loader  : {len(test_loader):,} batches ({len(test_dataset):,} câu)")

📦 Train Loader : 366,688 batches (23,468,005 câu)
📦 Val Loader   : 20,372 batches (1,303,778 câu)
📦 Test Loader  : 20,372 batches (1,303,779 câu)


## 9. Định nghĩa Kiến trúc Encoder (Bi-directional GRU)
Bộ mã hóa 2 chiều xuôi - ngược giúp nắm bắt toàn diện ngữ cảnh trước và sau của mỗi từ tiếng Anh.

In [9]:
class EncoderWithAttention(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(emb_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.dropout(self.embedding(src))
        encoder_states, hidden = self.rnn(embedded)
        # Nén trạng thái ẩn 2 chiều thành 1 chiều cho Decoder
        s_0 = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        return encoder_states, s_0

## 10. Định nghĩa Cơ chế Bahdanau Attention (Additive Attention)
Cơ chế tính điểm tập trung giữa trạng thái Decoder hiện tại và toàn bộ các từ của câu tiếng Anh nguồn.

In [10]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_states, mask=None):
        # decoder_hidden: [batch_size, hidden_dim]
        # encoder_states: [batch_size, src_len, hidden_dim * 2]
        s_proj = self.W_s(decoder_hidden).unsqueeze(1)
        h_proj = self.W_h(encoder_states)
        energy = torch.tanh(s_proj + h_proj)
        attention = self.v(energy).squeeze(2)  # [batch_size, src_len]
        
        # MASKING ATTENTION: Triệt tiêu điểm chú ý vào các token đệm <pad>
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e4)
            
        return F.softmax(attention, dim=1)

## 11. Định nghĩa Kiến trúc Decoder với Attention
Decoder kết hợp nhúng từ vựng hiện tại và Dynamic Context Vector từ Attention để dự đoán từ tiếng Việt tiếp theo.

In [11]:
class DecoderWithAttention(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, dropout=0.3, pad_idx=0, attention=None):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(emb_dim + hidden_dim * 2, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim + hidden_dim * 2 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_step, decoder_hidden, encoder_states, mask=None):
        input_step = input_step.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_step))

        # 1. Tính Dynamic Context Vector kèm mặt nạ Masking
        a = self.attention(decoder_hidden, encoder_states, mask=mask).unsqueeze(1)
        context = torch.bmm(a, encoder_states)

        # 2. Truyền qua GRU
        rnn_input = torch.cat((embedded, context), dim=2)
        output, next_hidden = self.rnn(rnn_input, decoder_hidden.unsqueeze(0))
        next_hidden = next_hidden.squeeze(0)

        # 3. Dự đoán từ tiếp theo
        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        context = context.squeeze(1)

        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))
        return prediction, next_hidden

## 12. Định nghĩa Lớp Bao Seq2Seq Hoàn Chỉnh
Điều phối toàn bộ luồng mã hóa, giải mã và cơ chế Teacher Forcing.

In [12]:
class Seq2SeqWithAttention(nn.Module):
    def __init__(self, encoder, decoder, pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.device = device

    def create_mask(self, src):
        # Tạo mặt nạ: True với từ thật, False với token <pad>
        mask = (src != self.pad_idx)
        return mask

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim

        # TẠO TRỰC TIẾP TRÊN GPU (device=self.device) - KHÔNG ĐI QUA RAM CPU
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size, device=self.device)
        
        # 1. Mã hóa câu nguồn
        encoder_states, decoder_hidden = self.encoder(src)

        # 2. Tạo mặt nạ Attention Mask để loại bỏ token <pad>
        mask = self.create_mask(src)

        input_token = tgt[:, 0]

        for t in range(1, tgt_len):
            prediction, decoder_hidden = self.decoder(input_token, decoder_hidden, encoder_states, mask=mask)
            outputs[:, t, :] = prediction
            
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            input_token = tgt[:, t] if teacher_force else top1

        return outputs

## 13. Khởi Tạo Mô Hình & Chuyển Lên GPU RTX 4070
Khởi tạo các tham số siêu phẳng, kiểm tra số lượng trọng số trainable và đưa mô hình lên GPU.

In [13]:
INPUT_DIM = len(vocab_en)
OUTPUT_DIM = len(vocab_vi)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HIDDEN_DIM = 512
ENC_DROPOUT = 0.35
DEC_DROPOUT = 0.35

encoder = EncoderWithAttention(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, ENC_DROPOUT, pad_idx=vocab_en.PAD_IDX)
attention = BahdanauAttention(HIDDEN_DIM)
decoder = DecoderWithAttention(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, DEC_DROPOUT, pad_idx=vocab_vi.PAD_IDX, attention=attention)

# Khởi tạo Seq2Seq kèm Attention Mask và đẩy thẳng lên GPU
model = Seq2SeqWithAttention(encoder, decoder, pad_idx=vocab_en.PAD_IDX, device=device).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("=" * 65)
print(f"🏗️ MÔ HÌNH SEQ2SEQ ATTENTION (CUDA OPTIMIZED):")
print(f"🔹 Từ vựng Tiếng Anh : {INPUT_DIM:,} từ")
print(f"🔹 Từ vựng Tiếng Việt: {OUTPUT_DIM:,} từ (Đã tối ưu chuẩn 32k)")
print(f"🔹 Tổng số tham số   : {count_parameters(model):,}")
print(f"🔹 Vị trí thiết bị    : {next(model.parameters()).device}")
print("=" * 65)

🏗️ MÔ HÌNH SEQ2SEQ ATTENTION (CUDA OPTIMIZED):
🔹 Từ vựng Tiếng Anh : 32,004 từ
🔹 Từ vựng Tiếng Việt: 32,004 từ (Đã tối ưu chuẩn 32k)
🔹 Tổng số tham số   : 80,201,988
🔹 Vị trí thiết bị    : cuda:0


## 14. Thiết Lập Hàm Mất Mát (Loss) & Bộ Tối Ưu (Optimizer)
Sử dụng `CrossEntropyLoss` kèm **`label_smoothing = 0.1`** (chống overfit cực mạnh) và `AdamW` với `weight_decay = 1e-4`.

In [ ]:
# Bỏ qua token PAD và thêm Label Smoothing 0.1
criterion = nn.CrossEntropyLoss(ignore_index=vocab_vi.PAD_IDX, label_smoothing=0.1)

# AdamW với Weight Decay kìm hãm trọng số không bị phình to
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

## 15. Kích Hoạt CUDA AMP GradScaler & Bộ Điều Chỉnh Tốc Độ Học
Sử dụng `torch.cuda.amp.GradScaler()` để điều khiển tính toán tự động trên các nhân **Tensor Cores (FP16)** và `ReduceLROnPlateau`.

In [ ]:
# Bộ điều chỉnh tỷ lệ Mixed Precision của CUDA Toolkit (AMP FP16)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# Bộ giảm Learning Rate tự động khi loss chững lại
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.6, patience=4
)

print(f"⚡ Đã khởi tạo CUDA GradScaler (Tensor Cores FP16 Enabled: {scaler.is_enabled()})")

⚡ Đã khởi tạo CUDA GradScaler (Tensor Cores FP16 Enabled: True)


## 16. Định Nghĩa Hàm Huấn Luyện 1 Epoch (train_epoch với CUDA AMP)
Hàm huấn luyện 1 epoch tích hợp `torch.cuda.amp.autocast()`, `scaler.scale()`, kẹp gradient và truyền dữ liệu bất đồng bộ `non_blocking = True`.

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, scaler, clip=1.0, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0

    for src, tgt in dataloader:
        # Tối ưu CUDA: Chuyển dữ liệu bất đồng bộ qua PCIe
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        optimizer.zero_grad()

        # TỰ ĐỘNG DÙNG NHÂN TENSOR CORES (FP16 MIXED PRECISION)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            output = model(src, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
            output_dim = output.shape[-1]
            output_flat = output[:, 1:].contiguous().view(-1, output_dim)
            tgt_flat = tgt[:, 1:].contiguous().view(-1)
            loss = criterion(output_flat, tgt_flat)

        # Lan truyền ngược với GradScaler để chống Underflow
        scaler.scale(loss).backward()
        
        # Bỏ tỷ lệ để kẹp gradient chống bùng nổ
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Cập nhật trọng số
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

## 17. Định Nghĩa Hàm Đánh Giá Xác Thực (evaluate_epoch)
Hàm đánh giá mô hình trên tập Validation (tắt Teacher Forcing, chạy với `torch.no_grad()` và `autocast`).

In [ ]:
def evaluate_epoch(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt in dataloader:
            src = src.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                output = model(src, tgt, teacher_forcing_ratio=0.0)
                output_dim = output.shape[-1]
                output_flat = output[:, 1:].contiguous().view(-1, output_dim)
                tgt_flat = tgt[:, 1:].contiguous().view(-1)
                loss = criterion(output_flat, tgt_flat)

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

## 18. Vòng Lặp Huấn Luyện Chính (Training Loop với Early Stopping)
Vòng lặp chính qua các Epochs, tự động lưu mô hình tốt nhất `best_model_full.pt` và tự động dừng sớm nếu Val Loss không cải thiện sau 6 epochs.

In [ ]:
N_EPOCHS = 10    
CLIP = 1.0
EARLY_STOP_PATIENCE = 5

best_valid_loss = float('inf')
model_save_path = "best_model_full.pt"
no_improve_epochs = 0

train_losses = []
val_losses = []
train_ppls = []
val_ppls = []

print("=" * 70)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN TOÀN DIỆN VỚI CUDA TOOLKIT & TENSOR CORES...")
print(f"⚡ Batch size hiện tại: {train_loader.batch_size} câu/batch (Đã tối ưu VRAM)")
print("=" * 70)

for epoch in range(N_EPOCHS):
    start_time = time.time()

    # 1. Huấn luyện và Đánh giá
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, clip=CLIP, teacher_forcing_ratio=0.5)
    val_loss = evaluate_epoch(model, val_loader, criterion)

    # 2. Cập nhật Learning Rate
    scheduler.step(val_loss)

    elapsed = time.time() - start_time
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)

    # 3. Tính Perplexity
    train_ppl = math.exp(min(train_loss, 50))
    val_ppl = math.exp(min(val_loss, 50))

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ppls.append(train_ppl)
    val_ppls.append(val_ppl)

    # 4. Kiểm tra và Lưu Checkpoint tốt nhất
    if val_loss < best_valid_loss:
        best_valid_loss = val_loss
        torch.save(model.state_dict(), model_save_path)
        saved_mark = "⭐ [Đã lưu Best Model]"
        no_improve_epochs = 0
    else:
        saved_mark = ""
        no_improve_epochs += 1

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{N_EPOCHS:02d} | {mins:02d}m{secs:02d}s | LR: {current_lr:.1e} | Train Loss: {train_loss:.4f} (PPL: {train_ppl:6.2f}) | Val Loss: {val_loss:.4f} (PPL: {val_ppl:6.2f}) {saved_mark}")

    # 5. Dừng sớm (Early Stopping)
    if no_improve_epochs >= EARLY_STOP_PATIENCE:
        print(f"🛑 Dừng sớm tại Epoch {epoch+1} vì Val Loss không giảm trong {EARLY_STOP_PATIENCE} epochs liên tiếp!")
        break

print("=" * 70)
print(f"🎉 Huấn luyện hoàn tất! Best Val Loss: {best_valid_loss:.4f}")

🚀 BẮT ĐẦU HUẤN LUYỆN TOÀN DIỆN VỚI CUDA TOOLKIT & TENSOR CORES...
⚡ Batch size hiện tại: 64 câu/batch (Đã tối ưu VRAM)


RuntimeError: value cannot be converted to type c10::Half without overflow

## 19. Vẽ Biểu Đồ Đánh Giá Tiến Trình Huấn Luyện (Loss & Perplexity)
Vẽ đồ thị theo dõi độ dốc hội tụ của Train Loss vs Validation Loss và Train PPL vs Validation PPL.

In [ ]:
plt.figure(figsize=(14, 5))

# Đồ thị Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', color='forestgreen', linewidth=2)
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='darkorange', linewidth=2)
plt.title('Biểu đồ Loss Toàn Diện (CUDA Accelerated)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss (CrossEntropy)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Đồ thị Perplexity
plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_ppls) + 1), train_ppls, label='Train PPL', color='forestgreen', linewidth=2)
plt.plot(range(1, len(val_ppls) + 1), val_ppls, label='Validation PPL', color='darkorange', linewidth=2)
plt.title('Biểu đồ Perplexity (PPL)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.yscale('log')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 20. Định Nghĩa Hàm Dịch Văn Bản Hoàn Chỉnh (Inference)
Hàm dịch câu tiếng Anh sang tiếng Việt, tự động giải mã các từ ghép có dấu gạch dưới `_` của PyVi thành văn bản tiếng Việt tự nhiên.

In [ ]:
def translate(sentence, model, vocab_src, vocab_tgt, max_len=60):
    model.eval()
    tokens = vocab_src.tokenizer(sentence)
    indices = [vocab_src.SOS_IDX] + [vocab_src.word2idx.get(t, vocab_src.UNK_IDX) for t in tokens] + [vocab_src.EOS_IDX]
    src_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            encoder_states, decoder_hidden = model.encoder(src_tensor)
            mask = model.create_mask(src_tensor)

    trg_indices = [vocab_tgt.SOS_IDX]
    for _ in range(max_len):
        prev_token = torch.tensor([trg_indices[-1]], dtype=torch.long).to(device)
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                output, decoder_hidden = model.decoder(prev_token, decoder_hidden, encoder_states, mask=mask)
                
        pred_token = output.argmax(1).item()
        if pred_token == vocab_tgt.EOS_IDX:
            break
        trg_indices.append(pred_token)

    # Giải mã và chuyển các từ ghép 'đọc_sách' thành 'đọc sách' tự nhiên
    decoded_raw = vocab_tgt.decode(trg_indices)
    decoded_clean = decoded_raw.replace("_", " ")
    return decoded_clean

## 21. Thử Nghiệm Dịch Thực Tế (Demo & Test)
Nạp lại trọng số tốt nhất `best_model_full.pt` và chạy thử nghiệm dịch các câu kiểm tra.

In [ ]:
# 1. Nạp lại mô hình tốt nhất
model.load_state_dict(torch.load("best_model_full.pt", map_location=device))
model.eval()

# 2. Danh sách câu mẫu
test_cases = [
    "I have a big passion for reading books in my free time.",
    "Every evening, I usually spend about thirty minutes reading a novel before going to bed.",
    "Reading is truly a wonderful way for me to balance my life.",
    "What are you talking about?",
    "Please stand behind the yellow line."
]

print("=" * 80)
print(f"{'CÂU TIẾNG ANH (EN)':<40} | {'BẢN DỊCH TIẾNG VIỆT (VI)'}")
print("=" * 80)
for sent in test_cases:
    vi_trans = translate(sent, model, vocab_en, vocab_vi)
    print(f"{sent:<40} | {vi_trans}")
print("=" * 80)